# RHCR Combined Comparison Plot

将 `exp_diff_sim_plan` 指定窗口组结果与 `rhcr_compare` 结果画在同一张图（LTF风格：均值折线 + 95%CI阴影）。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_rows', 200)


In [ ]:
# ===== 配置 =====
diff_root = Path('/home/shiqi/masterarbeit/RHCR/exp_diff_sim_plan')
learned_root = Path('/home/shiqi/masterarbeit/RHCR/exp_learned/rhcr_eval_runs')

def latest_summary_dir(root: Path) -> Path:
    candidates = sorted([d for d in root.iterdir() if d.is_dir() and (d / 'summary.csv').exists()])
    assert candidates, f'No result dirs with summary.csv found under {root}'
    return candidates[-1]

# 默认：自动使用最新结果目录
diff_dir = latest_summary_dir(diff_root)
cmp_dir = latest_summary_dir(learned_root)

# 手动指定时，取消注释并改成目标目录
# diff_dir = Path('/home/shiqi/masterarbeit/RHCR/exp_diff_sim_plan/<run_name>')
# cmp_dir  = Path('/home/shiqi/masterarbeit/RHCR/exp_learned/rhcr_eval_runs/<run_name>')

# 从 exp_diff_sim_plan 里选择多组(sim,plan)一起画
window_pairs = [(1,5), (1,10), (1,20), (5,10), (5,20)]

plot_title = 'RHCR Combined: learned + baseline(sim/plan variants)'
x_key='num_agents'
x_label='agent_num'
y_key='throughput_per_step'
fig_w=6.0
fig_h=5.0
line_w=2.0

diff_csv = diff_dir / 'summary.csv'
cmp_csv = cmp_dir / 'summary.csv'
assert diff_csv.exists(), diff_csv
assert cmp_csv.exists(), cmp_csv
print('diff_sim_plan:', diff_csv)
print('learned:', cmp_csv)


In [ ]:
d1 = pd.read_csv(diff_csv)
d2 = pd.read_csv(cmp_csv)

for d in (d1, d2):
    d['num_agents'] = pd.to_numeric(d['num_agents'], errors='coerce').astype('Int64')
    d['seed'] = pd.to_numeric(d['seed'], errors='coerce').astype('Int64')
    d['throughput_per_step'] = pd.to_numeric(d['throughput_per_step'], errors='coerce')

diff_parts = []
for sim_w, plan_w in window_pairs:
    sub = d1[(d1['status']=='ok') & (d1['simulation_window']==sim_w) & (d1['planning_window']==plan_w)].copy()
    sub['algorithm'] = f'baseline sim={sim_w}, plan={plan_w}'
    diff_parts.append(sub[['algorithm','num_agents','seed','throughput_per_step']])
e = pd.concat(diff_parts, ignore_index=True) if diff_parts else pd.DataFrame(columns=['algorithm','num_agents','seed','throughput_per_step'])

c = d2[(d2['status']=='ok') & (d2['mode']=='learned')].copy()
c['algorithm'] = 'learned cost sim=1, plan=5'
c = c[['algorithm','num_agents','seed','throughput_per_step']]

df = pd.concat([e, c], ignore_index=True)
display(df.head(12))
print('rows:', len(df))
print('algorithms:', sorted(df['algorithm'].dropna().unique()))


In [ ]:
agg=(df.groupby(['algorithm',x_key])[y_key]
      .agg(['mean','std','count'])
      .reset_index())
agg['sem']=agg['std']/np.sqrt(agg['count'].clip(lower=1))
agg['ci95']=1.96*agg['sem']
display(agg.sort_values([x_key,'algorithm']))


In [ ]:
# LTF 风格：均值折线 + 95%CI 阴影
plt.figure(figsize=(fig_w, fig_h))

hue_order = [f'baseline sim={s}, plan={p}' for s,p in window_pairs] + ['learned cost sim=1, plan=5']
# 统一颜色映射（throughput 与 compute-time 共用）
color_map = {
    'learned cost sim=1, plan=5': 'red',
    'baseline sim=1, plan=10': 'orange',
    'baseline sim=1, plan=20': 'green',
    'baseline sim=5, plan=10': 'saddlebrown',
    'baseline sim=5, plan=20': 'purple',
}
for algo in hue_order:
    sub = agg[agg['algorithm'] == algo].sort_values(x_key)
    if len(sub) == 0:
        continue
    x = sub[x_key].tolist()
    y = sub['mean'].tolist()
    c = sub['ci95'].fillna(0).tolist()
    color = color_map.get(algo, None)
    plt.plot(x, y, marker='o', linewidth=line_w, label=str(algo), color=color)
    lo = [yy - cc for yy, cc in zip(y, c)]
    hi = [yy + cc for yy, cc in zip(y, c)]
    plt.fill_between(x, lo, hi, alpha=0.18, color=color)

plt.title(plot_title)
plt.xlabel(x_label)
plt.ylabel(y_key)
xt = sorted([int(v) for v in agg[x_key].dropna().unique().tolist()])
if xt:
    plt.xticks(xt)
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
out_pdf = cmp_dir / 'combined_diff_sim_plan_vs_learned.pdf'
plt.savefig(out_pdf)
print('Saved:', out_pdf)
plt.show()


In [ ]:
# 计算时间图：覆盖 throughput 图中的所有组合，颜色保持一致
import re

def parse_runlog_solver_times(log_path: Path):
    if not log_path.exists():
        return []
    times = []
    for ln in log_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        if ':Succeed,' not in ln:
            continue
        try:
            payload = ln.split(':Succeed,', 1)[1]
            t = float(payload.split(',', 1)[0])
            times.append(t)
        except Exception:
            pass
    return times

diff_df = pd.read_csv(diff_csv)
cmp  = pd.read_csv(cmp_csv)

# 与 throughput 图完全一致的算法集合
target_algos = [f'baseline sim={s}, plan={p}' for s,p in window_pairs] + ['learned cost sim=1, plan=5']

sel_diff = diff_df[(diff_df['status']=='ok')].copy()
sel_diff['algorithm'] = sel_diff.apply(lambda r: f"baseline sim={int(r['simulation_window'])}, plan={int(r['planning_window'])}", axis=1)
sel_diff = sel_diff[sel_diff['algorithm'].isin(target_algos)].copy()

sel_cmp = cmp[(cmp['status']=='ok') & (cmp['mode']=='learned')].copy()
sel_cmp['algorithm'] = 'learned cost sim=1, plan=5'

rows = []
for _, r in pd.concat([sel_diff, sel_cmp], ignore_index=True).iterrows():
    run_dir = Path(str(r['run_dir']))
    ts = parse_runlog_solver_times(run_dir / 'run.log')
    if len(ts) == 0:
        continue
    rows.append({
        'algorithm': r['algorithm'],
        'num_agents': int(r['num_agents']),
        'seed': int(r['seed']),
        'mean_solver_time': float(np.mean(ts)),
    })

rt = pd.DataFrame(rows)
assert not rt.empty, 'No runtime data parsed from run.log'

agg_rt = (rt.groupby(['algorithm', 'num_agents'])['mean_solver_time']
          .agg(['mean','std','count'])
          .reset_index())
agg_rt['sem'] = agg_rt['std'] / np.sqrt(agg_rt['count'].clip(lower=1))
agg_rt['ci95'] = 1.96 * agg_rt['sem']
display(agg_rt.sort_values(['num_agents','algorithm']))

plt.figure(figsize=(fig_w, fig_h))
order = target_algos
# 与 throughput 图一致的颜色
color_map_rt = {
    'learned cost sim=1, plan=5': 'red',
    'baseline sim=1, plan=10': 'orange',
    'baseline sim=1, plan=20': 'green',
    'baseline sim=5, plan=10': 'saddlebrown',
    'baseline sim=5, plan=20': 'purple',
}
for algo in order:
    sub = agg_rt[agg_rt['algorithm']==algo].sort_values('num_agents')
    if len(sub)==0:
        continue
    x = sub['num_agents'].tolist()
    y = sub['mean'].tolist()
    c = sub['ci95'].fillna(0).tolist()
    color = color_map_rt.get(algo, None)
    plt.plot(x, y, marker='o', linewidth=line_w, label=algo, color=color)
    lo = [yy-cc for yy,cc in zip(y,c)]
    hi = [yy+cc for yy,cc in zip(y,c)]
    plt.fill_between(x, lo, hi, alpha=0.18, color=color)

plt.title('Compute Time Comparison (mean per planning call)')
plt.xlabel('agent_num')
plt.ylabel('time (s)')
xt = sorted([int(v) for v in agg_rt['num_agents'].dropna().unique().tolist()])
if xt:
    plt.xticks(xt)
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
out_pdf_rt = cmp_dir / 'combined_compute_time_all_curves.pdf'
plt.savefig(out_pdf_rt)
print('Saved:', out_pdf_rt)
plt.show()
